In [1]:
pip install openai

In [2]:
import pandas as pd
from openai import OpenAI

client = OpenAI()

In [6]:
#Load csv through pandas to manipulate data in python.
notes = pd.read_csv("notes.csv")
print(notes.info())
notes.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   note_id           8 non-null      int64  
 1   claim             7 non-null      object 
 2   source_file       6 non-null      object 
 3   page_number       6 non-null      float64
 4   evidence_snippet  7 non-null      object 
 5   status            8 non-null      object 
dtypes: float64(1), int64(1), object(4)
memory usage: 516.0+ bytes
None


,note_id,claim,source_file,page_number,evidence_snippet,status
0,1,Climate change accelerates glacier melting,ipcc_report_2023.pdf,14.0,Glaciers have lost 30% mass since 1980,confirmed
1,2,Remote work increases productivity by 13%,stanford_wfh_study.pdf,NaN,Output measured over 9 month period,unverified
2,3,Sugar consumption linked to cognitive decline,NaN,22.0,High fructose diets showed memory impairment i...,needs review
3,4,Renewable energy costs dropped 89% in a decade,irena_report.pdf,5.0,Solar PV cost fell from $0.38 to $0.04 per kWh,confirmed
4,5,Sleep deprivation affects immune response,sleep_medicine_review.pdf,31.0,NaN,unverified


In [12]:
#Creates function that outputs whether a claim is supported in a text, whether that claim has a page number, and if the page number is valid. 
#List "issues" is loaded with what is specifically identified by the function as an issue.
#Variable "status" is deemed error if page number is not valid, deemed warning when page number, claim, and/or soure file is missing. 
def check_row(row):
    issues = []
    if pd.isna(row["claim"]):
        issues.append("Warning: Claim is Missing")
    if pd.isna(row["source_file"]):
        issues.append("Warning: Source File is missing")
    if pd.isna(row["page_number"]):
        issues.append("Warning: Page Number is missing")
    elif row["page_number"] <= 0:
        issues.append("Error: Page Number is not valid")
    if any("Error" in i for i in issues):
        status = "Error"
    elif any("Warning" in i for i in issues):
        status = "Warning"
    else:
        status = "Pass"
    return status, issues

results = []
#Applies function to the csv file. Each row is evaluated according to the criteria in the function, and based on the issues assessed by the function.
#A new dataframe is created that gives each "note_id"(claim) its status according to the function and the issues that gave it its status. (Claude helped me to write this code and understand what its doing).
for _, row in notes.iterrows():
    status, issues = check_row(row)
    results.append([row ["note_id"], status, "; ".join(issues)])

results_notes = pd.DataFrame(results, columns=["note_id", "status", "issues"])
results_notes

,note_id,status,issues
0,1,Pass,
1,2,Warning,Warning: Page Number is missing
2,3,Warning,Warning: Source File is missing
3,4,Pass,
4,5,Pass,
5,6,Warning,Warning: Claim is Missing
6,7,Pass,
7,8,Warning,Warning: Source File is missing; Warning: Page...


In [15]:
#This calls the OpenAI agent to evaluate the table created, create a summary, output issues and respond with suggestions.
#The agent is given the instructions to act like a provenance checker for claims that are not evaluated as pass to provide the
#summary, issues and suggestions.
flag = results_notes[results_notes["status"] != "Pass"].to_string(index=False)

prompt = f"""
You are a provenance checker tasked with ensuring data quality.
The issues found from a research notes CSV are:
{flag}
Respond only in JSON with no extra text, like this:
{{
    "Summary": "Your 2-3 sentence summary here",
    "Issues": "issue 1, issue 2",
    "Suggestion": "improvement suggestion here"
}}
"""
response = client.chat.completions.create(
    model = "gpt-4.1",
    messages=[{"role": "user", "content": prompt}]
)
#This gives the output structure for clarity using json. "response.choices[0].message.content is used to get most specifically the first in a list of ChatGPT's
#response, and not many elements of the list/non important parts of the response (images, emojis, etc.).
import json
nonjson = response.choices[0].message.content
json = json.loads(nonjson)

print("Summary:", json["Summary"])
print("Issues:", json["Issues"])
print("Suggestion:", json["Suggestion"])

Summary: Several entries in the research notes CSV contain missing information, particularly related to page numbers, source files, and claims. These omissions may compromise data completeness and reliability.
Issues: Missing page numbers, missing source files, missing claims
Suggestion: Ensure all required fields—such as source file, page number, and claim—are filled in for every entry, and consider adding data validation checks to prevent incomplete submissions.
